# Pathway Classification

Classifies sampled charge-transfer pathways across a grain boundary into
primary transport archetypes. All computation lives in the `gbctp` package;
this notebook only configures inputs, runs the pipeline, and plots the
results. See the repository README for the method summary.

In [ ]:
from pathlib import Path

from gbctp.pipeline import PipelineConfig, run_pipeline

# ---------------------------------------------------------------------------
# INPUT DATA (not included in this repository)
#
# ROOT_DIR must contain one coupling-time-series CSV per sampled pathway.
# Expected format of each CSV (see gbctp/io.py:read_path_csv for details):
#
#   sites,1276,1834,2210,...           <- site IDs along the path, source -> target
#   time,"cpl(1,2)","cpl(2,3)",...     <- header: "time" + one column per dimer
#   0.0,0.021,-0.034,...               <- coupling time series (signed, arbitrary units)
#   0.5,0.019,-0.031,...
#   ...
#
# CSV_PATTERN is a glob (relative to ROOT_DIR) that matches exactly one such
# file per pathway, e.g. if every pathway has its own subdirectory:
#   ROOT_DIR/<pathway_id>/TRAJ1/path_couplings_timeseries.csv
# ---------------------------------------------------------------------------
ROOT_DIR = Path("<path/to/pathway_csvs>")  # TODO: point this at your data
OUTPUT_DIR = Path("<path/to/output_dir>")  # TODO: where to write result CSVs
CSV_PATTERN = "*/TRAJ1/path_couplings_timeseries.csv"  # TODO: adjust to your directory layout

config = PipelineConfig(
    root_dir=ROOT_DIR,
    output_dir=OUTPUT_DIR,
    csv_pattern=CSV_PATTERN,
    # See gbctp.pipeline.PipelineConfig for every option (quantile
    # thresholds, representative-path feature columns, label-merging
    # rules, etc.) -- defaults reproduce the settings used in the paper.
)


## Run the classification pipeline

In [ ]:
result = run_pipeline(config)

print("Primary label counts:")
print(result.summary_primary.to_string(index=False))


## Plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from gbctp.io import read_path_csv
from gbctp.descriptors import per_dimer_vectors_abs
from gbctp.representatives import DEFAULT_FEATURE_COLS_FOR_MEDOID

plt.style.use("default")
sns.set_palette("husl")


### Medoid selection (PCA view)

2D PCA projection of one primary label's feature space, showing the class
centroid and the pathway(s) selected as representative ("medoid").

In [ ]:
def plot_medoid_selection_for_label(
    df, label_value, primary_col="label_primary", feature_cols=None,
    n_select=3, within_class_standardize=True, annotate=True,
):
    if feature_cols is None:
        feature_cols = DEFAULT_FEATURE_COLS_FOR_MEDOID

    sub = df[df[primary_col] == label_value].dropna(subset=feature_cols).copy()
    if len(sub) < 2:
        raise ValueError(f"Not enough complete rows in '{label_value}' (need >= 2)")

    X = sub[feature_cols].to_numpy(float)
    Xs = StandardScaler().fit_transform(X) if within_class_standardize else X

    centroid = Xs.mean(axis=0)
    dists = np.linalg.norm(Xs - centroid, axis=1)
    chosen_idx = np.argsort(dists)[: min(n_select, len(sub))]

    pca = PCA(n_components=2)
    X2 = pca.fit_transform(Xs)
    c2 = pca.transform(centroid.reshape(1, -1))[0]

    plt.figure(figsize=(9, 6))
    plt.scatter(X2[:, 0], X2[:, 1], alpha=0.7, marker="o", label="Paths")
    plt.scatter([c2[0]], [c2[1]], marker="*", s=200, color="red", label="Centroid")
    plt.scatter(
        X2[chosen_idx, 0], X2[chosen_idx, 1],
        s=140, marker="o", edgecolors="black", linewidths=2, label="Selected medoids",
    )
    for rank, idx_i in enumerate(chosen_idx, start=1):
        plt.plot([c2[0], X2[idx_i, 0]], [c2[1], X2[idx_i, 1]], "k--", linewidth=1, alpha=0.5)
        if annotate:
            plt.text(X2[idx_i, 0], X2[idx_i, 1], f"#{rank}", fontsize=10, fontweight="bold")

    plt.title(f"Medoid selection: '{label_value}'\n(PCA projection of feature space)")
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


# Change label_to_plot to inspect a different primary label.
if len(result.df_primary) > 0:
    available_labels = result.df_primary["label_primary"].unique()
    label_to_plot = available_labels[0]
    plot_medoid_selection_for_label(result.df_primary, label_value=label_to_plot, n_select=10)


### Representative-path coupling profiles

J (coupling), C (coherence) and E (effective link) along the path, for one
representative pathway per primary label.

In [ ]:
def plot_representative_path_profiles(rep_df, max_plots=4):
    primary_labels = rep_df["label_primary"].unique()
    n_labels = min(len(primary_labels), max_plots)

    fig, axes = plt.subplots(n_labels, 3, figsize=(14, 4 * n_labels))
    if n_labels == 1:
        axes = axes.reshape(1, -1)

    for i, label in enumerate(primary_labels[:n_labels]):
        rows = rep_df[rep_df["label_primary"] == label]
        if len(rows) == 0:
            continue

        row = rows.iloc[0]
        sites, time, X, _ = read_path_csv(Path(row["path"]))
        J, C, A2, Astd, E = per_dimer_vectors_abs(X)
        dimer_idx = np.arange(1, len(J) + 1)

        axes[i, 0].plot(dimer_idx, J, "o-", color="steelblue")
        axes[i, 0].set_ylabel("J (time-avg |coupling|)")
        axes[i, 0].set_title(f"{label}\nCoupling profile")
        axes[i, 0].grid(alpha=0.3)

        axes[i, 1].plot(dimer_idx, C, "o-", color="coral")
        axes[i, 1].set_ylabel("C (coherence)")
        axes[i, 1].set_title("Coherence profile")
        axes[i, 1].grid(alpha=0.3)
        axes[i, 1].axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="C=0.5")
        axes[i, 1].legend()

        axes[i, 2].plot(dimer_idx, E, "o-", color="green")
        axes[i, 2].set_ylabel("E (J x C)")
        axes[i, 2].set_title("Effective coupling profile")
        axes[i, 2].grid(alpha=0.3)

        if i == n_labels - 1:
            for ax in axes[i]:
                ax.set_xlabel("Dimer position along path")

    plt.tight_layout()
    plt.show()


if len(result.rep_paths) > 0:
    plot_representative_path_profiles(result.rep_paths, max_plots=4)


### Descriptor distributions by primary label

Violin plots of key order-aware descriptors, split by primary label.

In [ ]:
def plot_descriptor_distributions(df, descriptors=("E_dip", "C_dip", "rE", "RE")):
    available = [d for d in descriptors if d in df.columns]
    if not available:
        print("No descriptor columns found for plotting")
        return

    fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 5))
    if len(available) == 1:
        axes = [axes]

    for ax, desc in zip(axes, available):
        labels_list = list(df["label_primary"].unique())
        plot_data = [df.loc[df["label_primary"] == lab, desc].dropna() for lab in labels_list]

        ax.violinplot(plot_data, positions=range(len(labels_list)), showmeans=True, showmedians=True)
        ax.set_xticks(range(len(labels_list)))
        ax.set_xticklabels(labels_list, rotation=45, ha="right")
        ax.set_ylabel(desc)
        ax.set_title(f"{desc} distribution")
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


if len(result.df_primary) > 0:
    plot_descriptor_distributions(result.df_primary)


## Optional: 3D bottleneck visualization

Plots one representative pathway in 3D, colored by effective link strength
and annotated with the bottleneck location. Requires:

- `pip install gbctp[geometry]` (installs MDAnalysis)
- a `.gro` structure file whose residue numbers match the `site_i`/`site_j`
  IDs in the pathway CSVs (e.g. an MD-averaged geometry)

Skipped automatically if either is unavailable.

In [ ]:
GRO_FILE = Path("<path/to/averaged_structure.gro>")  # TODO: set this to run this cell

try:
    from gbctp.geometry import read_gro_file
    HAVE_MDANALYSIS = True
except ImportError:
    HAVE_MDANALYSIS = False
    print("MDAnalysis not installed -- skipping 3D visualization. "
          "Install with: pip install gbctp[geometry]")

if HAVE_MDANALYSIS and not GRO_FILE.exists():
    print(f"GRO_FILE not found: {GRO_FILE} -- skipping 3D visualization.")


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3D projection)


def plot_path_3d_with_bottleneck(path_row, gro_file):
    com_coords = read_gro_file(gro_file)

    jce_file = Path(path_row["path"]).with_suffix(".JCE_vectors.csv")
    if not jce_file.exists():
        print(f"JCE vectors file not found: {jce_file}")
        return

    jce_data = pd.read_csv(jce_file)
    sites = list(jce_data["site_i"]) + [jce_data["site_j"].iloc[-1]]

    path_coords, valid = [], []
    for idx, site in enumerate(sites):
        if site in com_coords:
            path_coords.append(com_coords[site])
            valid.append(idx)
    if len(path_coords) < 2:
        print("Not enough valid coordinates to plot path")
        return
    path_coords = np.array(path_coords)

    E_values = jce_data["E_JxC"].to_numpy(float)
    E_min, E_max = E_values.min(), E_values.max()
    E_norm = (E_values - E_min) / (E_max - E_min) if E_max > E_min else np.ones_like(E_values)
    bottleneck_idx = int(np.argmin(E_values))

    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(path_coords[:, 0], path_coords[:, 1], path_coords[:, 2],
            "o-", color="gray", alpha=0.3, linewidth=1, markersize=4)

    for i in range(len(jce_data)):
        if i < len(path_coords) - 1:
            color = plt.cm.RdYlGn(E_norm[i])
            ax.plot(*zip(path_coords[i], path_coords[i + 1]), "-", color=color, linewidth=3, alpha=0.8)

    ax.scatter(*path_coords[0], color="green", s=200, marker="o",
               edgecolors="black", linewidths=2, label="Source", zorder=10)
    ax.scatter(*path_coords[-1], color="blue", s=200, marker="s",
               edgecolors="black", linewidths=2, label="Target", zorder=10)
    if bottleneck_idx < len(path_coords) - 1:
        ax.scatter(*path_coords[bottleneck_idx], color="red", s=300, marker="X",
                   edgecolors="darkred", linewidths=3, label="Bottleneck", zorder=15)
        ax.scatter(*path_coords[bottleneck_idx + 1], color="red", s=300, marker="X",
                   edgecolors="darkred", linewidths=3, zorder=15)

    ax.set_xlabel("X (nm)")
    ax.set_ylabel("Y (nm)")
    ax.set_zlabel("Z (nm)")
    ax.set_title(f"3D path visualization: {path_row.get('label_primary', 'Unknown')}")
    ax.legend(loc="upper right")

    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=plt.Normalize(vmin=E_min, vmax=E_max))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=5)
    cbar.set_label("Effective transport E (J x C)", rotation=270, labelpad=20)

    plt.tight_layout()
    plt.show()


if HAVE_MDANALYSIS and GRO_FILE.exists():
    for _, row in result.rep_paths.iterrows():
        plot_path_3d_with_bottleneck(row, GRO_FILE)
